# Research Paper Domain Classification — Portfolio Code Sample

This notebook contains a concise selection of the implemented code from the original project.

**Included**
- arXiv data loading and preparation
- multi-label target preparation
- TF-IDF + Linear SVM
- BERT embeddings + Logistic Regression
- validation-based hyperparameter tuning
- model evaluation
- LIME explainability
- YAKE keyword extraction

The full academic project also explored additional components. This public code sample focuses only on the implemented components most relevant to NLP and machine learning.


## 1. Setup
Run the installation line only if the required packages are not already available.


In [ ]:
# Uncomment if needed:
# !pip -q install sentence-transformers lime yake

import os
import ast
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

MIN_TEXT_CHARS = 30
TOP_K_KEYWORDS = 10
MODEL_BERT = "bert-base-nli-mean-tokens"


## 2. Load the arXiv Dataset

Expected columns: `terms`, `titles`, and `abstracts`.


In [ ]:
def read_arxiv_csv(path):
    try:
        return pd.read_csv(path)
    except pd.errors.ParserError:
        return pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip",
            sep=",",
            quotechar='"'
        )

DATA_PATH = "arxiv_data_210930-054931.csv"  # update if needed

df_raw = read_arxiv_csv(DATA_PATH)

print("Raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head(3)


## 3. Data Preparation

The original implementation:
- combines title and abstract into one text field,
- parses multi-label academic-domain targets,
- removes invalid/short records,
- removes exact duplicate texts.


In [ ]:
def parse_terms(x):
    if pd.isna(x):
        return []

    if isinstance(x, (list, tuple, set)):
        return [str(t).strip() for t in x if str(t).strip()]

    s = str(x).strip()
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            value = ast.literal_eval(s)
            if isinstance(value, (list, tuple, set)):
                return [str(t).strip() for t in value if str(t).strip()]
        except Exception:
            pass

    if "," in s:
        return [t.strip() for t in s.split(",") if t.strip()]

    return [s]


def build_text(title, abstract):
    title = "" if pd.isna(title) else str(title).strip()
    abstract = "" if pd.isna(abstract) else str(abstract).strip()
    return f"{title}. {abstract}".strip()


df = df_raw.rename(columns={"titles": "title", "abstracts": "abstract"}).copy()
df["text"] = df.apply(
    lambda row: build_text(row.get("title", ""), row.get("abstract", "")),
    axis=1
)
df["labels"] = df["terms"].apply(parse_terms)

df = df[df["text"].str.len() >= MIN_TEXT_CHARS]
df = df[df["labels"].apply(len) > 0]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Cleaned rows:", len(df))
print("Unique labels:", len({label for labels in df["labels"] for label in labels}))

df[["text", "labels"]].head(3)


## 4. Train / Validation / Test Split

The project uses an **80% / 10% / 10%** split with random seed **42**.


In [ ]:
texts = df["text"].tolist()
labels = df["labels"].tolist()

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    texts, Y, test_size=0.20, random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print("Number of labels:", len(mlb.classes_))


In [ ]:
def multilabel_metrics(y_true, y_pred):
    return {
        "subset_accuracy": float(accuracy_score(y_true, y_pred)),
        "hamming_loss": float(hamming_loss(y_true, y_pred)),
        "precision_micro": float(
            precision_score(y_true, y_pred, average="micro", zero_division=0)
        ),
        "recall_micro": float(
            recall_score(y_true, y_pred, average="micro", zero_division=0)
        ),
        "f1_micro": float(
            f1_score(y_true, y_pred, average="micro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
    }


## 5. Model 1 — TF-IDF + Linear SVM


In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)


## 6. Model 2 — BERT Embeddings + Logistic Regression

Pretrained BERT sentence embeddings are used as contextual text representations.


In [ ]:
from sentence_transformers import SentenceTransformer

bert_embedder = SentenceTransformer(MODEL_BERT)

def encode_texts(model, texts, batch_size=32):
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True
    )

X_train_bert = encode_texts(bert_embedder, X_train)
X_val_bert = encode_texts(bert_embedder, X_val)
X_test_bert = encode_texts(bert_embedder, X_test)


## 7. Hyperparameter Tuning

The regularization parameter `C` is selected using validation **Micro-F1**.


In [ ]:
def tune_svm_C(X_train, y_train, X_val, y_val, C_grid=(0.1, 0.3, 1, 3, 10)):
    rows = []
    best_f1, best_model, best_C = -1, None, None

    for C in C_grid:
        clf = OneVsRestClassifier(LinearSVC(C=C))
        clf.fit(X_train, y_train)

        pred = clf.predict(X_val)
        score = f1_score(y_val, pred, average="micro", zero_division=0)

        rows.append({"C": C, "val_f1_micro": float(score)})

        if score > best_f1:
            best_f1, best_model, best_C = score, clf, C

    return best_model, best_C, pd.DataFrame(rows)


def tune_lr_C(X_train, y_train, X_val, y_val, C_grid=(0.1, 0.3, 1, 3, 10)):
    rows = []
    best_f1, best_model, best_C = -1, None, None

    for C in C_grid:
        clf = OneVsRestClassifier(
            LogisticRegression(C=C, max_iter=3000, n_jobs=-1)
        )
        clf.fit(X_train, y_train)

        pred = clf.predict(X_val)
        score = f1_score(y_val, pred, average="micro", zero_division=0)

        rows.append({"C": C, "val_f1_micro": float(score)})

        if score > best_f1:
            best_f1, best_model, best_C = score, clf, C

    return best_model, best_C, pd.DataFrame(rows)


svm_best, svm_best_C, svm_tuning = tune_svm_C(
    X_train_tfidf, y_train, X_val_tfidf, y_val
)

bert_best, bert_best_C, bert_tuning = tune_lr_C(
    X_train_bert, y_train, X_val_bert, y_val
)

print("Best SVM C:", svm_best_C)
print("Best BERT+LR C:", bert_best_C)

display(svm_tuning.sort_values("val_f1_micro", ascending=False))
display(bert_tuning.sort_values("val_f1_micro", ascending=False))


## 8. Evaluation


In [ ]:
pred_test_svm = svm_best.predict(X_test_tfidf)
pred_test_bert = bert_best.predict(X_test_bert)

svm_metrics = multilabel_metrics(y_test, pred_test_svm)
bert_metrics = multilabel_metrics(y_test, pred_test_bert)

results = pd.DataFrame([
    {"model": "TF-IDF + Linear SVM", **svm_metrics},
    {"model": "BERT Embeddings + Logistic Regression", **bert_metrics},
]).sort_values("f1_micro", ascending=False)

results


## 9. Explainability — LIME

LIME is used to inspect which words contribute positively or negatively to a selected domain prediction.


In [ ]:
from lime.lime_text import LimeTextExplainer

label_support = y_train.sum(axis=0)
label_idx = int(np.argmax(label_support))
label_name = mlb.classes_[label_idx]

explainer = LimeTextExplainer(
    class_names=[f"not {label_name}", label_name]
)

def proba_one_label(texts):
    embeddings = bert_embedder.encode(
        texts,
        show_progress_bar=False,
        batch_size=32
    )
    probabilities = bert_best.predict_proba(embeddings)[:, label_idx]
    return np.vstack([1 - probabilities, probabilities]).T

sample_text = X_test[0]

explanation = explainer.explain_instance(
    sample_text,
    proba_one_label,
    num_features=12
)

print("Explaining label:", label_name)
explanation.as_pyplot_figure()


## 10. Keyword Extraction — YAKE


In [ ]:
import yake

keyword_extractor = yake.KeywordExtractor(
    lan="en",
    n=1,
    top=TOP_K_KEYWORDS
)

def extract_keywords(text):
    return [
        keyword
        for keyword, score in keyword_extractor.extract_keywords(text)
    ]

extract_keywords(X_test[0])


## Portfolio Scope

This notebook intentionally contains a concise selection of the implemented NLP/ML code.

The incomplete SDG-mapping/deployment placeholder and duplicate experimental visualization cells from the working notebook are not included in this public portfolio version.
